# 📚 DocuMind — RAG Pipeline Demo

This notebook demonstrates the full RAG pipeline: document ingestion, hybrid retrieval, and LLM-based generation.

**Runtime**: GPU recommended for embedding speed, but CPU works fine.

**Setup**: Add your HuggingFace token to Colab Secrets (`HF_TOKEN`).

In [ ]:
# Install dependencies
!pip install -q sentence-transformers chromadb rank-bm25 pypdf python-docx requests

In [ ]:
# Clone the repo (replace with your GitHub URL after pushing)
# !git clone https://github.com/YOUR_USERNAME/smart-doc-assistant.git
# %cd smart-doc-assistant

# OR mount Google Drive if you saved the project there
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/smart-doc-assistant

In [ ]:
import os

# Load HF token from Colab Secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN', '')

print('Token loaded:', bool(HF_TOKEN))

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.document_processor import process_document, Chunk
from src.embedder import Embedder
from src.vector_store import VectorStore
from src.retriever import HybridRetriever
from src.generator import Generator
from src.rag_pipeline import RAGPipeline

## 1. Initialize Embedder
Uses `all-MiniLM-L6-v2` — fast, high quality, 384-dimensional embeddings.

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

embedder = Embedder(device=device)
print(f'Embedding dimension: {embedder.dim}')

## 2. Build the RAG Pipeline

In [ ]:
vector_store = VectorStore(persist_dir='./demo_chroma')
generator = Generator(hf_token=HF_TOKEN)
pipeline = RAGPipeline(embedder, vector_store, generator, top_k=5)

## 3. Ingest Sample Documents

In [ ]:
# Ingest sample text passages
docs = [
    ("""The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need' 
    by Vaswani et al. It relies entirely on attention mechanisms, dispensing with recurrence and 
    convolutions entirely. The model uses multi-head self-attention and position-wise feed-forward 
    networks. Transformers have become the dominant architecture for NLP tasks.""", "transformers_paper"),
    
    ("""Retrieval-Augmented Generation (RAG) combines a retrieval system with a language model. 
    First introduced by Lewis et al. (2020), RAG retrieves relevant documents from an external 
    knowledge base and conditions the language model generation on these documents. This approach 
    reduces hallucination and enables models to answer questions about private or recent data.""", "rag_paper"),
    
    ("""BM25 (Best Match 25) is a probabilistic ranking function used in information retrieval. 
    It improves on TF-IDF by accounting for document length normalization and term frequency saturation. 
    BM25 is particularly effective for keyword-based queries and complements dense semantic search 
    in hybrid retrieval systems.""", "bm25_overview"),
    
    ("""Sentence transformers are pre-trained models that map sentences to dense vector representations. 
    The 'all-MiniLM-L6-v2' model achieves strong performance on semantic textual similarity tasks 
    while being small enough to run on CPU. It produces 384-dimensional embeddings that can be 
    compared using cosine similarity.""", "sentence_transformers"),
]

for text, source in docs:
    pipeline.ingest_text(text, source_name=source)

print(f'Total chunks in store: {pipeline._vector_store.count()}')

## 4. Query the System

In [ ]:
questions = [
    "What is RAG and how does it reduce hallucination?",
    "What is BM25 and why is it useful in retrieval?",
    "How does the Transformer architecture work?",
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print('-' * 60)
    
    response = pipeline.query(question)
    print(f"A: {response.answer}")
    print(f"\nLatency: {response.latency_ms:.0f}ms | Chunks retrieved: {len(response.source_chunks)}")
    print("\nTop source:")
    if response.source_chunks:
        top = response.source_chunks[0]
        print(f"  Score: {top.get('rrf_score', top.get('score', 0)):.4f}")
        print(f"  Text: {top['text'][:200]}...")

## 5. Retrieval Analysis — Semantic vs Hybrid

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

query = "dense vector representations for sentences"

# Pure semantic search
query_emb = embedder.embed_single(query)
semantic_results = vector_store.search(query_emb, k=4)

# Hybrid search
hybrid_results = pipeline._retriever.retrieve(query, k=4)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, results, title in zip(
    axes,
    [semantic_results, hybrid_results],
    ['Semantic Search', 'Hybrid (BM25 + Semantic + RRF)']
):
    scores = [r.get('rrf_score', r.get('score', 0)) for r in results]
    labels = [r['text'][:40] + '...' for r in results]
    bars = ax.barh(range(len(scores)), scores, color='steelblue', alpha=0.8)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Score')
    ax.set_title(title)
    ax.invert_yaxis()

plt.suptitle(f'Query: "{query}"', fontweight='bold')
plt.tight_layout()
plt.savefig('retrieval_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to retrieval_comparison.png')

## 6. Embedding Space Visualization (t-SNE)

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

texts = [d[0] for d in docs]
labels = [d[1] for d in docs]

embeddings = embedder.embed(texts)

# Add query embedding
query_text = "How does RAG work?"
query_emb = embedder.embed_single(query_text)
all_embeddings = np.vstack([embeddings, query_emb])
all_labels = labels + ["QUERY"]

tsne = TSNE(n_components=2, random_state=42, perplexity=2)
reduced = tsne.fit_transform(all_embeddings)

plt.figure(figsize=(10, 7))
colors = plt.cm.Set1(np.linspace(0, 1, len(all_labels)))

for i, (label, color) in enumerate(zip(all_labels, colors)):
    marker = '*' if label == 'QUERY' else 'o'
    size = 300 if label == 'QUERY' else 150
    plt.scatter(reduced[i, 0], reduced[i, 1], c=[color], s=size, marker=marker, zorder=5)
    plt.annotate(label, (reduced[i, 0], reduced[i, 1]), textcoords='offset points',
                 xytext=(5, 5), fontsize=10)

plt.title(f't-SNE Embedding Space\nQuery: "{query_text}"', fontweight='bold')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig('embedding_space.png', dpi=150, bbox_inches='tight')
plt.show()